# Auditoría de Drive — E3

Ejecuta todas las celdas. Al final obtienes un bloque de texto que puedes copiar y pegar a Claude.

No borra ni modifica nada — solo lista.

In [ ]:
from google.colab import drive
import os
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    print('Drive ya montado.')
else:
    drive.mount('/content/drive')
    print('Drive montado.')

In [ ]:
import os
from pathlib import Path
from datetime import datetime

DRIVE = '/content/drive/MyDrive'

# ── Rutas a escanear ─────────────────────────────────────────
# Añade aquí cualquier ruta adicional que quieras revisar
RUTAS = [
    # Datos de entrenamiento
    f'{DRIVE}/Datos_E2_E3/General',
    f'{DRIVE}/Datos_E2_E3/E3',
    f'{DRIVE}/Datos_E2_E3',
    # Nombres alternativos que se han usado en distintos momentos
    f'{DRIVE}/Datos_E3',
    f'{DRIVE}/TFM',
    f'{DRIVE}/TFM_datos',
    f'{DRIVE}/shapenet_limpias',
    f'{DRIVE}/objaverse_limpias',
    f'{DRIVE}/fantastic_breaks_procesado',
]

def tamanyo_legible(n_bytes):
    for u in ['B','KB','MB','GB']:
        if n_bytes < 1024: return f'{n_bytes:.1f} {u}'
        n_bytes /= 1024
    return f'{n_bytes:.1f} TB'

def escanear_carpeta(ruta, max_depth=3, _depth=0):
    """Devuelve lista de líneas descriptivas."""
    p = Path(ruta)
    lineas = []
    if not p.exists():
        return [f'  [NO EXISTE] {ruta}']
    try:
        hijos = sorted(p.iterdir())
    except PermissionError:
        return [f'  [SIN PERMISO] {ruta}']

    archivos = [h for h in hijos if h.is_file()]
    carpetas = [h for h in hijos if h.is_dir()]

    # Resumen de archivos en este nivel
    if archivos:
        por_ext = {}
        total_bytes = 0
        for f in archivos:
            try: sz = f.stat().st_size
            except: sz = 0
            total_bytes += sz
            ext = f.suffix.lower() or '(sin ext)'
            por_ext[ext] = por_ext.get(ext, 0) + 1
        ext_str = ', '.join(f'{v}x{k}' for k,v in sorted(por_ext.items()))
        ind = '  ' * (_depth + 1)
        lineas.append(f'{ind}📄 {len(archivos)} archivos [{ext_str}] — {tamanyo_legible(total_bytes)}')

    # Subcarpetas
    for c in carpetas:
        ind = '  ' * (_depth + 1)
        try:
            n_hijos = len(list(c.iterdir()))
        except: n_hijos = '?'
        # Tamaño de archivos directos
        try:
            arch_dir = [f for f in c.iterdir() if f.is_file()]
            sz_dir = sum(f.stat().st_size for f in arch_dir)
            sz_str = tamanyo_legible(sz_dir) if arch_dir else ''
        except: sz_str = ''

        lineas.append(f'{ind}📁 {c.name}/  ({n_hijos} elementos{(", "+sz_str) if sz_str else ""})')
        if _depth < max_depth - 1:
            lineas.extend(escanear_carpeta(c, max_depth, _depth + 1))

    return lineas


# ── Escaneo ───────────────────────────────────────────────────
bloques = []
vistas = set()

for ruta in RUTAS:
    p = Path(ruta)
    # Evitar escanear una carpeta padre si ya escaneamos una hija
    if not p.exists():
        bloques.append(f'\n[NO EXISTE] {ruta}')
        continue
    # Resolver ruta real
    try: real = str(p.resolve())
    except: real = ruta
    if real in vistas:
        bloques.append(f'\n[YA VISTO] {ruta} → {real}')
        continue
    vistas.add(real)

    bloques.append(f'\n{"="*60}')
    bloques.append(f'📂 {ruta}')
    bloques.append('='*60)
    bloques.extend(escanear_carpeta(ruta, max_depth=3))

for b in bloques:
    print(b)

In [ ]:
# ── Detalle de carpetas de modelos y resultados ───────────────
# Lista los checkpoints guardados por experimento con su tamaño y fecha

from pathlib import Path
from datetime import datetime
import os

DRIVE = '/content/drive/MyDrive'
BASE_E3 = f'{DRIVE}/Datos_E2_E3/E3/Raquel'

def fecha_mod(p):
    try: return datetime.fromtimestamp(p.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
    except: return '?'

def tamanyo_legible(n):
    for u in ['B','KB','MB','GB']:
        if n < 1024: return f'{n:.0f} {u}'
        n /= 1024
    return f'{n:.1f} TB'

print('=' * 70)
print('CHECKPOINTS DE MODELOS')
print('=' * 70)

modelos_dir = Path(f'{BASE_E3}/modelos')
if modelos_dir.exists():
    for exp in sorted(modelos_dir.iterdir()):
        if not exp.is_dir(): continue
        ckpts = sorted(exp.glob('*.pt'))
        total = sum(f.stat().st_size for f in ckpts if f.exists())
        print(f'\n  📁 {exp.name}/  ({tamanyo_legible(total)} total)')
        for c in ckpts:
            try: sz = tamanyo_legible(c.stat().st_size)
            except: sz = '?'
            print(f'    {c.name:25s}  {sz:>8}  {fecha_mod(c)}')
else:
    print(f'  [NO EXISTE] {modelos_dir}')

print()
print('=' * 70)
print('RESULTADOS (metricas.json)')
print('=' * 70)

import json
resultados_dir = Path(f'{BASE_E3}/resultados')
if resultados_dir.exists():
    for exp in sorted(resultados_dir.iterdir()):
        if not exp.is_dir(): continue
        m_path = exp / 'metricas.json'
        archivos = list(exp.iterdir())
        if m_path.exists():
            try:
                m = json.loads(m_path.read_text())
                print(f'\n  📁 {exp.name}/')
                print(f'    version={m.get("version")}  modelo={m.get("modelo")}')
                print(f'    datasets={m.get("datasets")}  pares={m.get("n_pares_total")}  test={m.get("n_test")}')
                print(f'    epochs={m.get("epochs")}  best_epoch={m.get("best_epoch")}')
                print(f'    CD-L1={m.get("cd_l1")}  F-Score={m.get("f_score")}')
                print(f'    notas={m.get("notas")}')
                otros = [f.name for f in archivos if f.name != 'metricas.json']
                if otros: print(f'    otros: {otros}')
            except Exception as e:
                print(f'  📁 {exp.name}/  [ERROR leyendo metricas.json: {e}]')
        else:
            print(f'\n  📁 {exp.name}/  (sin metricas.json — archivos: {[f.name for f in archivos]})')
else:
    print(f'  [NO EXISTE] {resultados_dir}')

In [ ]:
# ── Resumen final para copiar a Claude ───────────────────────
# Copia TODO el output de esta celda y pégalo en el chat

from pathlib import Path
from datetime import datetime
import json

DRIVE = '/content/drive/MyDrive'
BASE_E3 = f'{DRIVE}/Datos_E2_E3/E3/Raquel'
BASE_GEN = f'{DRIVE}/Datos_E2_E3/General'

lineas = []
lineas.append('=== AUDITORÍA DRIVE ===')
lineas.append(f'Fecha: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

def tamanyo(n):
    for u in ['B','KB','MB','GB']:
        if n < 1024: return f'{n:.0f}{u}'
        n /= 1024
    return f'{n:.1f}TB'

# --- Datasets en General ---
lineas.append('')
lineas.append('--- DATASETS (General/) ---')
gen = Path(BASE_GEN)
if gen.exists():
    for d in sorted(gen.iterdir()):
        if d.is_dir():
            try:
                archivos = list(d.iterdir())
                npys = [f for f in archivos if f.suffix == '.npy']
                plys = [f for f in archivos if f.suffix == '.ply']
                otros_ext = set(f.suffix for f in archivos if f.suffix not in ('.npy','.ply'))
                sz = sum(f.stat().st_size for f in archivos if f.is_file())
                det = f'{len(npys)}x.npy' if npys else ''
                det += f' {len(plys)}x.ply' if plys else ''
                det += f' otros:{otros_ext}' if otros_ext else ''
                lineas.append(f'  {d.name}/  {tamanyo(sz)}  [{det.strip()}]')
            except Exception as e:
                lineas.append(f'  {d.name}/  [ERROR: {e}]')
        else:
            lineas.append(f'  {d.name}  (archivo)')
else:
    lineas.append(f'  [NO EXISTE] {BASE_GEN}')

# --- Modelos ---
lineas.append('')
lineas.append('--- MODELOS (E3/Raquel/modelos/) ---')
mod = Path(f'{BASE_E3}/modelos')
if mod.exists():
    for exp in sorted(mod.iterdir()):
        if exp.is_dir():
            ckpts = sorted(exp.glob('*.pt'))
            names = [c.name for c in ckpts]
            sz = sum(c.stat().st_size for c in ckpts if c.exists())
            lineas.append(f'  {exp.name}/  {tamanyo(sz)}  {names}')
else:
    lineas.append(f'  [NO EXISTE] {mod}')

# --- Resultados ---
lineas.append('')
lineas.append('--- RESULTADOS (E3/Raquel/resultados/) ---')
res = Path(f'{BASE_E3}/resultados')
if res.exists():
    for exp in sorted(res.iterdir()):
        if exp.is_dir():
            m_path = exp / 'metricas.json'
            if m_path.exists():
                try:
                    m = json.loads(m_path.read_text())
                    lineas.append(f'  {exp.name}/  CD={m.get("cd_l1")}  F={m.get("f_score")}'
                                  f'  ep={m.get("best_epoch")}  test={m.get("n_test")}')
                except:
                    lineas.append(f'  {exp.name}/  [metricas.json ilegible]')
            else:
                archs = [f.name for f in exp.iterdir()]
                lineas.append(f'  {exp.name}/  [sin metricas.json]  archivos={archs}')
else:
    lineas.append(f'  [NO EXISTE] {res}')

# --- Resto de E3/Raquel ---
lineas.append('')
lineas.append('--- RESTO E3/Raquel/ ---')
e3r = Path(BASE_E3)
if e3r.exists():
    for d in sorted(e3r.iterdir()):
        if d.name in ('modelos','resultados'): continue
        if d.is_dir():
            try:
                n = len(list(d.iterdir()))
                lineas.append(f'  {d.name}/  ({n} elementos)')
            except:
                lineas.append(f'  {d.name}/  [error]')
        else:
            lineas.append(f'  {d.name}  (archivo)')
else:
    lineas.append(f'  [NO EXISTE] {BASE_E3}')

lineas.append('')
lineas.append('=== FIN AUDITORÍA ===')

print('\n'.join(lineas))
print()
print('↑ Copia todo el texto entre las líneas === y pégalo en Claude.')